<a href="https://colab.research.google.com/github/bsheese/387/blob/main/20_how_networks_learn/20_0_PyTorch_Basics/20_0_3_Gradient_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Networks Learn: Part 3
## Gradient Descent

---

## What This Notebook Is About

20_0_2 showed how to get the slope of the loss for every weight. This notebook uses those slopes to find good weights. The method is **gradient descent**: work out every weight's slope, move every weight a small step downhill, and repeat. It's what `optimizer.step()` does in the training loops you've run, and it's how every network in this course is trained.

To see each step clearly we use the smallest possible model: a straight line that predicts a penguin's body mass from the length of its flippers. A line has just two parameters, a **weight** $w$ (its steepness) and a **bias** $b$ (where it crosses the axis), so it's a single neuron with one input and no activation. Small as it is, it trains exactly the way a large network does.

Straight lines have one advantage for learning: there's a formula that gives the best line directly, which is what scikit-learn's `LinearRegression` uses. So we can check that gradient descent finds the right answer. Networks have no such formula, which is why they're trained by gradient descent.

**What you will learn:**
1. What the loss looks like as a weight changes: the loss "bowl"
2. How to take one step of gradient descent by hand
3. How repeating the step finds the same line as the formula
4. Why the step size (the learning rate) matters so much
5. What `optimizer.step()` does

**Optional reference.** If you took CS 377, you have seen these ideas before in [17_0_5](https://github.com/bsheese/cs377/blob/main/17_regression_crossval/17_0_Preliminaries/17_0_5_Rsquared.ipynb). If you haven't, those notebooks cover the same ideas on other datasets, in more depth. You don't need them for this course.

In [ ]:
import os
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LinearRegression

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

penguins = pd.read_csv(fetch('penguins.csv')).dropna(subset=['flipper_length_mm', 'body_mass_g'])
flipper = torch.tensor(penguins['flipper_length_mm'].to_numpy(), dtype=torch.float32)
mass = torch.tensor(penguins['body_mass_g'].to_numpy(), dtype=torch.float32)
print(len(flipper), 'penguins')

plt.figure(figsize=(6, 4))
plt.scatter(flipper, mass, alpha=0.5)
plt.xlabel('flipper length (mm)')
plt.ylabel('body mass (g)')
plt.title('Palmer penguins')
plt.show()

Penguins with longer flippers tend to be heavier, and the relationship looks roughly like a straight line.

### Putting both measurements on the same scale

Flipper lengths are around 200 and body masses around 4,000, so the slopes for $w$ and $b$ would be on very different scales, and no single step size would suit both. The usual fix is to **standardize** each measurement: subtract its average and divide by its standard deviation (a measure of how spread out it is). Afterward both have an average of 0 and most values fall between about −2 and 2. A value of 1 means "one standard deviation above average".

In [ ]:
x = (flipper - flipper.mean()) / flipper.std()
y = (mass - mass.mean()) / mass.std()
print(f'x: average {x.mean():.3f}, standard deviation {x.std():.3f}, range {x.min():.2f} to {x.max():.2f}')
print(f'y: average {y.mean():.3f}, standard deviation {y.std():.3f}, range {y.min():.2f} to {y.max():.2f}')

The model predicts $\hat{y} = w \cdot x + b$ in these standardized units, and the loss is the **mean squared error** (MSE): the average of the squared differences between predictions and actual values.

In [ ]:
def mse(w, b):
    return ((w * x + b - y) ** 2).mean()

---

## Section 1: The Loss Bowl

With only two parameters we can look at the loss directly. Because both measurements are standardized, the best bias is 0, so fix $b = 0$ for the moment and compute the loss for many values of $w$:

In [ ]:
w_values = torch.linspace(-0.5, 2.0, 200)
bowl = torch.stack([mse(w, 0.0) for w in w_values])
best = w_values[bowl.argmin()].item()

def draw_bowl(ax):
    ax.plot(w_values, bowl, lw=2)
    ax.set_xlabel('weight w')
    ax.set_ylabel('loss (MSE)')

fig, ax = plt.subplots(figsize=(7, 4))
draw_bowl(ax)
ax.scatter([0.0], [mse(0.0, 0.0)], color='C3', s=80, zorder=3, label='start: w = 0')
ax.scatter([best], [bowl.min()], color='C2', s=80, zorder=3, label=f'bottom: w = {best:.2f}')
ax.legend()
ax.set_title('The loss for each value of w (b = 0)')
plt.show()

The loss forms a bowl. Its bottom, at $w$ near 0.87, is the best weight: the line that fits the penguins most closely. We'll start from $w = 0$, a flat line that predicts the average mass for every penguin, and see if gradient descent can find the bottom on its own.

The bowl also shows what a slope means here. On the left side the bowl goes downhill as $w$ increases, so the slope is negative and $w$ should go up. On the right side the slope is positive and $w$ should go down. At the bottom the slope is 0. A network's loss depends on millions of weights, so it can't be drawn, but each weight's slope says the same thing about which way is downhill for that weight.

---

## Section 2: One Step by Hand

First, the slope at the start. Here it is with the nudge test from 20_0_2, and then with autograd:

In [ ]:
h = 0.001
nudge_w = (mse(0.0 + h, 0.0) - mse(0.0 - h, 0.0)) / (2 * h)
nudge_b = (mse(0.0, 0.0 + h) - mse(0.0, 0.0 - h)) / (2 * h)
print(f'nudge test:  slope for w {nudge_w.item():.3f}, slope for b {nudge_b.item():.3f}')

w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
loss = mse(w, b)
loss.backward()
print(f'autograd:    slope for w {w.grad.item():.3f}, slope for b {b.grad.item():.3f}')

The two methods agree. The slope for $w$ is about −1.74: negative, so increasing $w$ lowers the loss. The slope for $b$ is 0, because $b = 0$ is already the best bias.

A step of gradient descent moves each parameter **against** its slope, by the slope times a small number called the **learning rate**:

$$w_{\text{new}} = w - \text{learning rate} \times \text{slope for } w$$

Subtracting means a negative slope makes $w$ go up and a positive slope makes it go down, which is downhill either way. Multiplying by the slope means the step is bigger where the bowl is steep and smaller near the bottom, where the slope is close to 0.

The update is wrapped in `torch.no_grad()` because changing a weight isn't part of the calculation we want PyTorch to record.

In [ ]:
learning_rate = 0.1
print(f'before: w = {w.item():.4f}, b = {b.item():.4f}, loss = {mse(w, b).item():.4f}')

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

print(f'after:  w = {w.item():.4f}, b = {b.item():.4f}, loss = {mse(w, b).item():.4f}')

One step moved $w$ from 0 to about 0.17, and the loss went down.

---

## Section 3: Repeating the Step

Training repeats that step: clear the old slopes, compute the loss, get the new slopes with `.backward()`, take a step. The loop below runs 100 steps from the start and records the loss and $w$ after each one. (Each step here uses all the penguins at once, so one step is also one epoch.)

In [ ]:
def gradient_descent(learning_rate, steps=100):
    w = torch.tensor(0.0, requires_grad=True)
    b = torch.tensor(0.0, requires_grad=True)
    losses, ws = [], []
    for step in range(steps):
        if w.grad is not None:              # 1. clear last step's slopes
            w.grad.zero_()
            b.grad.zero_()
        loss = mse(w, b)                    # 2. the loss with the current weights
        loss.backward()                     # 3. the slopes
        with torch.no_grad():               # 4. a step downhill
            w -= learning_rate * w.grad
            b -= learning_rate * b.grad
        losses.append(loss.item())
        ws.append(w.item())
    return w.item(), b.item(), losses, ws

w_final, b_final, losses, ws = gradient_descent(0.1)
print(f'after 100 steps: w = {w_final:.4f}, b = {b_final:.4f}, loss = {mse(w_final, b_final).item():.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(losses)
ax1.set_xlabel('step')
ax1.set_ylabel('loss (MSE)')
ax1.set_title('Loss after each step (learning rate 0.1)')

draw_bowl(ax2)
first = [0.0] + ws[:12]
ax2.scatter(first, [mse(v, 0.0).item() for v in first], color='C3', zorder=3)
ax2.set_title('Where w was during the first 12 steps')
plt.tight_layout()
plt.show()

The loss falls quickly at first and then levels off. On the bowl, the first steps are large because the slope is steep, and they get shorter as $w$ approaches the bottom, where the slope is nearly 0. Nobody told the loop where the bottom was. It found it by following the slope.

---

## Section 4: The Same Line as the Formula

scikit-learn's `LinearRegression` calculates the best line directly from the data. To compare, convert our line from standardized units back to grams and millimeters. (A prediction of $\hat{y}$ standard deviations above average mass is $\hat{y}$ times the standard deviation of mass, plus the average mass, and likewise for flipper length.)

In [ ]:
formula = LinearRegression().fit(flipper.numpy().reshape(-1, 1), mass.numpy())

slope_gd = w_final * mass.std().item() / flipper.std().item()
intercept_gd = mass.mean().item() + b_final * mass.std().item() - slope_gd * flipper.mean().item()

print(f'formula:          {formula.coef_[0]:.2f} g per mm, intercept {formula.intercept_:.0f} g')
print(f'gradient descent: {slope_gd:.2f} g per mm, intercept {intercept_gd:.0f} g')

line_x = np.linspace(170, 235, 50)
plt.figure(figsize=(6, 4))
plt.scatter(flipper, mass, alpha=0.4)
plt.plot(line_x, formula.coef_[0] * line_x + formula.intercept_, 'g-', lw=4, label='formula')
plt.plot(line_x, slope_gd * line_x + intercept_gd, 'r--', lw=2, label='gradient descent')
plt.xlabel('flipper length (mm)')
plt.ylabel('body mass (g)')
plt.legend()
plt.show()

The two lines are the same to within rounding: each extra millimeter of flipper goes with about 50 more grams of penguin. For a straight line, the formula is the better tool, since it's exact and instant. For a network there is no formula, and gradient descent, which only needs slopes, works the same way whether there are two weights or eleven million.

---

## Section 5: The Learning Rate

The learning rate sets the size of every step. Here are three choices, each run for 100 steps:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, lr in zip(axes, [0.001, 0.1, 1.05]):
    w_end, b_end, losses_lr, ws_lr = gradient_descent(lr)
    ax.plot(losses_lr)
    ax.set_yscale('log')
    ax.set_xlabel('step')
    ax.set_ylabel('loss (log scale)')
    ax.set_title(f'learning rate {lr}: final w = {w_end:.3g}')
plt.tight_layout()
plt.show()

- **Too small (0.001):** every step is tiny. After 100 steps $w$ has only reached about 0.16, less than a fifth of the way to the bottom, and the loss is still far above its lowest value. It would get there eventually, but it would take thousands of steps.
- **About right (0.1):** the loss reaches the bottom within a few dozen steps.
- **Too large (1.05):** each step overshoots the bottom and lands farther up the other side of the bowl, where the slope is steeper, so the next step is bigger still. The loss grows instead of shrinking, and $w$ ends up at a huge value. This is called **diverging**. (The loss axis is on a log scale, so each gridline is ten times the one below it.)

The middle plot also shows a loss that levels off rather than reaching 0. The bottom of the bowl isn't zero, because no straight line passes through every penguin.

There's no formula for the right learning rate. In practice people try a few values that differ by factors of 10 (0.001, 0.01, 0.1) and watch the loss curve. A curve that barely moves means the rate is too small, and one that jumps around or grows means it's too large.

---

## Section 6: What `optimizer.step()` Does

The update we wrote by hand, `w -= learning_rate * w.grad` for every parameter, is exactly what PyTorch's plain gradient descent optimizer, `torch.optim.SGD`, does in `optimizer.step()`. `optimizer.zero_grad()` clears every parameter's `.grad`. Here is the same training written with an optimizer:

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.1)

for step in range(100):
    optimizer.zero_grad()
    loss = mse(w, b)
    loss.backward()
    optimizer.step()

print(f'with torch.optim.SGD: w = {w.item():.4f}, b = {b.item():.4f}')
print(f'by hand:              w = {w_final:.4f}, b = {b_final:.4f}')

The same answer. (SGD stands for "stochastic gradient descent". "Stochastic" refers to using a random batch of the data for each step, as 19_0_1 did with batches of 32; here every step used all of the data.)

The training loops in 19_0_1 and 19_1_1 used a different optimizer, **Adam**. Adam still moves every weight against its slope, but it adjusts the step size for each weight separately based on that weight's recent slopes, which usually makes the learning rate less fussy to choose. Unit 21 compares the two.

---

## Summary

- **Gradient descent** finds good weights by repeating one step: compute the loss, get every weight's slope with `.backward()`, and move every weight against its slope by the slope times the **learning rate**.
- For a model with one weight, the loss as a function of the weight is a **bowl**. Gradient descent starts somewhere on its side and follows the slope down. Steps are large where the bowl is steep and small near the bottom.
- Standardizing the inputs (subtract the average, divide by the standard deviation) keeps the slopes of different parameters on a similar scale, so one learning rate suits all of them.
- On the penguins, gradient descent finds the same line as `LinearRegression`'s formula. Networks have no formula, so gradient descent is how they're trained.
- A learning rate that's **too small** barely moves; one that's **too large** overshoots and **diverges**. Choose it by trying values a factor of 10 apart and watching the loss curve.
- `optimizer.zero_grad()` clears the slopes and `optimizer.step()` takes the step. `torch.optim.SGD` is exactly the update written by hand here; Adam is a variant that adjusts each weight's step size.

**Next:** 20_1 builds networks out of layers of neurons with PyTorch's `nn.Module`, and trains one with a hidden layer to solve the XOR problem from 19_1_4.